# [P2] Q2 A + B: data audit and cleaning

**Owner:** Part 2  
**Temporary part notebook.** At integration it is merged into `Q2_laptop_analysis.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q2a` (audit), `t_q2b` (cleaning and new variables, with validation).

Decisions to read on the team board first (board ids): `q_data q_dups q_issues` (A), `os_map st_type rare validate` (B), and the shared `st_extra`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


# Q2 A: Data audit

Before analysing the laptop data we investigate its structure and quality. This notebook section applies the team decisions below.

| Decision | What we chose |
|---|---|
| `q_data` | The dataset is a copy in `data/laptop_price_dataset.csv` (no space in the name). The raw file is never modified. |
| `q_dups` | Near-duplicate rows are kept and documented; at the end we check that the conclusions hold without them. |
| `q_issues` | Which problems we address and which we only document (see the register in section 7). |
| `q_audit` | The audit contains: a variables table, a statement of what a row is, data type checks, range and unit checks, and consistency checks between columns. |
| `g_lang`, `g_nb` | English, and every step is explained in a text cell before its code. |

## 0. Setup

We import the libraries and read the file with `load_raw()`. This function only reads: no column is changed, no row is dropped. Keeping the raw table untouched lets us always compare against the original.

In [ ]:
# HE: ייבוא ספריות. numpy היא ספרייה לחישובים על מספרים. pandas היא ספרייה לעבודה עם טבלאות.
# HE: מקובל לקצר את השמות: np ו-pd.
import numpy as np
import pandas as pd

# HE: load_raw היא הפונקציה שכתבנו בקובץ q2_clean.py. היא רק קוראת את קובץ הנתונים.
from q2_clean import load_raw

# HE: שתי השורות הבאות משנות רק איך הטבלאות מוצגות על המסך (רוחב ומספר עמודות מקסימלי).
# HE: הן לא משנות את הנתונים עצמם.
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# HE: קוראים את הנתונים למשתנה raw (גולמי). לעולם לא משנים את raw עצמו: כל ניקוי יעשה על עותק.
raw = load_raw()

# HE: raw.shape נותן זוג מספרים: (כמה שורות, כמה עמודות).
print("shape (rows, columns):", raw.shape)

# HE: raw.head(3) מציג את שלוש השורות הראשונות, כדי לראות איך הנתונים נראים.
# HE: בתא במחברת, השורה האחרונה מוצגת אוטומטית במסך.
raw.head(3)

## 1. What does one row represent?

Decision (`q_audit`, row definition): we state in one sentence what a row is, because it determines what counts as one observation.

**Each row is a laptop listing: one configuration (brand, model, screen, CPU, RAM, storage, GPU, operating system, weight) together with one price in euros.** A row is not necessarily a unique model: the same configuration can appear several times with different prices. We check this in section 4.

## 2. Variables table

Decision (`q_audit`, variables table): one row per column with its pandas data type, its kind, the number of different values, the number of missing values, and an example. The *kind* is assigned by us from the meaning of the column, and `assert` makes sure that no column is left unclassified.

- **numeric**: values we can compute with (price, weight, RAM...).
- **categorical**: labels or types, with no arithmetic meaning (company, operating system...).
- **structured text**: text that contains several pieces of information in one string and has to be parsed before use (for example `Memory` holds both capacity and storage type).

In [ ]:
# HE: אנחנו מסווגים כל עמודה לפי המשמעות שלה, ולא לפי מה שפייתון חושב עליה.
# HE: NUMERIC: מספרים שאפשר לחשב איתם. CATEGORICAL: תוויות (חברה, מערכת הפעלה).
# HE: STRUCTURED_TEXT: טקסט שמכיל כמה פרטים במחרוזת אחת (למשל Memory: "256GB SSD + 1TB HDD").
NUMERIC = ["Inches", "CPU_Frequency (GHz)", "RAM (GB)", "Weight (kg)", "Price (Euro)"]
CATEGORICAL = ["Company", "TypeName", "CPU_Company", "GPU_Company", "OpSys"]
STRUCTURED_TEXT = ["Product", "ScreenResolution", "CPU_Type", "Memory", "GPU_Type"]

# HE: kind הוא מילון (dictionary): לכל שם עמודה יש סוג. הכתיב {c: "numeric" for c in NUMERIC}
# HE: אומר "לכל עמודה c ברשימה NUMERIC, שייך לה את הערך numeric". update מוסיף למילון עוד זוגות.
kind = {c: "numeric" for c in NUMERIC}
kind.update({c: "categorical" for c in CATEGORICAL})
kind.update({c: "structured text" for c in STRUCTURED_TEXT})

# HE: assert הוא בדיקה: אם התנאי לא נכון, התוכנית נעצרת עם ההודעה שבסוף.
# HE: כאן בודקים שכל עמודה בטבלה סווגה, ושלא כתבנו שם עמודה לא קיים. כך לא נשכח עמודה בשקט.
assert set(kind) == set(raw.columns), "every column must be classified exactly once"

# HE: בונים טבלת סיכום עם שורה לכל עמודה:
# HE: dtype = סוג הנתונים לפי פייתון. n_unique = כמה ערכים שונים יש בעמודה.
# HE: n_missing = כמה תאים ריקים (isna מזהה ריקים, sum סופר אותם).
# HE: example = ערך לדוגמה מהשורה הראשונה. reindex(raw.columns) שומר על סדר העמודות של הקובץ.
variables = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "kind": pd.Series(kind).reindex(raw.columns),
    "n_unique": raw.nunique(),
    "n_missing": raw.isna().sum(),
    "example": raw.iloc[0],
})
variables

**What the table shows**

- The table has 1,275 rows and 15 columns: 5 numeric, 5 categorical and 5 structured-text columns.
- **No column has missing values** (`n_missing` is 0 everywhere).
- The numeric columns already have numeric types (`float64` or `int64`), so they can be used in calculations as they are. The text columns have the pandas text type (`str`).
- Several text columns have many different values: `Product` (618), `GPU_Type` (106), `CPU_Type` (93), `ScreenResolution` (40) and `Memory` (39). They mix several pieces of information in one string, and `Memory` in particular has to be parsed to get the storage capacity and the storage type (section 2 of the cleaning notebook, task Q2 B).
- `Price (Euro)` has 791 different values among 1,275 rows: prices repeat, which is expected when the same configuration is listed more than once. We look at this in section 4.

## 3. Data types: are they appropriate for the analysis?

Decision (`q_audit`, data types): we check that the columns we want to compute with are really numeric, and we list the text columns that must be converted before use, with evidence from the data.

In [ ]:
# HE: בודקים שהעמודות המספריות באמת בסוג מספרי. אם עמודה כזו הייתה נקראת כטקסט, לא היה אפשר לחשב איתה.
# HE: is_numeric_dtype מחזירה True אם סוג העמודה מספרי. המילון numeric_ok שומר תשובה לכל עמודה.
numeric_ok = {c: pd.api.types.is_numeric_dtype(raw[c]) for c in NUMERIC}
print(numeric_ok)
assert all(numeric_ok.values()), "every numeric column must have a numeric dtype"

# HE: עכשיו אוספים ראיות לכך שעמודות הטקסט מערבבות כמה פרטים במחרוזת אחת.
# HE: str.contains בודקת לכל שורה אם הטקסט מכיל תבנית, ו-sum סופר בכמה שורות זה נכון.
# HE: regex=False אומר "חפש את הטקסט כמו שהוא" (חשוב לסימן + שיש לו משמעות מיוחדת בביטוי רגולרי).
mem = raw["Memory"]
res = raw["ScreenResolution"]
evidence = pd.Series({
    "Memory: two drives (contains '+')": int(mem.str.contains("+", regex=False).sum()),
    "Memory: capacity in TB": int(mem.str.contains("TB", regex=False).sum()),
    "Memory: contains a double space": int(mem.str.contains("  ", regex=False).sum()),
    "Memory: Flash Storage": int(mem.str.contains("Flash", regex=False).sum()),
    "Memory: Hybrid": int(mem.str.contains("Hybrid", regex=False).sum()),
    "ScreenResolution: has a WxH resolution": int(res.str.contains(r"\d+x\d+").sum()),
    "ScreenResolution: Touchscreen": int(res.str.contains("Touchscreen", regex=False).sum()),
    "ScreenResolution: IPS panel": int(res.str.contains("IPS", regex=False).sum()),
})
evidence

Columns to convert, with the new variables they will produce (decision `st_extra` and the cleaning decisions `os_map` and `st_type`; the code is written in task Q2 B):

In [ ]:
# HE: זו רשימה ידנית (כתבנו אותה בעצמנו, אין כאן חישוב): איזו עמודה גולמית צריך לפרק,
# HE: למה, ואילו עמודות חדשות היא תיתן. היא מסכמת את ההחלטות שקיבלנו בלוח.
# HE: pd.DataFrame(רשימה של מילונים) בונה טבלה, כשכל מילון הוא שורה.
to_convert = pd.DataFrame([
    {"raw column": "Memory", "problem": "capacity and storage type in one string, 1-2 drives, GB and TB",
     "new variables": "Storage_type, Storage_GB_total, has_SSD"},
    {"raw column": "ScreenResolution", "problem": "resolution and panel features in one string",
     "new variables": "Screen_pixels, Touchscreen"},
    {"raw column": "CPU_Type", "problem": "processor level and model together",
     "new variables": "CPU_tier (Core i3 / i5 / i7 / other)"},
    {"raw column": "OpSys", "problem": "same system under two names (macOS and Mac OS X)",
     "new variables": "OpSys_clean, OS_family"},
    {"raw column": "GPU_Company", "problem": "needed as a yes/no for a dedicated graphics card",
     "new variables": "Dedicated_GPU"},
    {"raw column": "Price (Euro)", "problem": "right-skewed, multiplicative effects",
     "new variables": "log_price"},
])
to_convert

**What section 3 shows**

- All five numeric columns already have numeric types, so no conversion is needed for them.
- `Memory` mixes capacity and storage type: 208 laptops have two drives, 455 use TB units, 71 use Flash Storage and 10 use Hybrid. The same 208 strings contain a double space, so the parser in Q2 B must handle double spaces and both GB and TB.
- `ScreenResolution` contains a `WxH` resolution in all 1,275 rows, so the number of pixels can be extracted for every laptop. It also holds panel features: 188 touchscreens and 357 IPS panels.

## 4. Missing values and duplicates

Decisions: `q_audit` (missing values) and `q_dups` (near-duplicate rows are kept and documented). We look for missing values in three ways: real empty cells, "hidden" missing values (blank text, stray spaces) and impossible numbers (zero or negative).

In [ ]:
# HE: חסרים "אמיתיים": תאים ריקים. isna() מזהה ריקים, sum() סופר אותם, ושוב sum() מסכם את כל הטבלה.
print("empty cells in the whole table:", int(raw.isna().sum().sum()))

# HE: חסרים "מוסתרים": טקסט ריק או רווחים מיותרים שלא נראים לעין.
# HE: str.strip() מוחקת רווחים בתחילת ובסוף המחרוזת. אם הטקסט שונה אחרי strip, היו שם רווחים מיותרים.
text_cols = CATEGORICAL + STRUCTURED_TEXT
hidden = pd.DataFrame({
    "blank_text": {c: int((raw[c].str.strip() == "").sum()) for c in text_cols},
    "leading_or_trailing_space": {c: int((raw[c] != raw[c].str.strip()).sum()) for c in text_cols},
    "double_space_inside": {c: int(raw[c].str.contains("  ", regex=False).sum()) for c in text_cols},
})
print("text columns with any hidden problem:")
print(hidden[(hidden > 0).any(axis=1)])

# HE: מספרים בלתי אפשריים: משקל, מחיר וגודל מסך חייבים להיות חיוביים.
print("non-positive values per numeric column:")
print((raw[NUMERIC] <= 0).sum())

In [ ]:
# HE: כפילויות מדויקות: שורות זהות בכל העמודות. duplicated() מסמנת שורה שכבר הופיעה קודם.
print("exact duplicate rows:", int(raw.duplicated().sum()))

# HE: כמעט-כפילויות: שורות זהות בכל העמודות חוץ מהמחיר (אותה תצורה, מחיר שונה).
# HE: key_cols הן כל העמודות חוץ מהמחיר. subset=key_cols אומר "השווה רק לפי העמודות האלה".
# HE: keep=False מסמן את כל השורות בקבוצה (ולא רק את הכפולה), כדי שנראה את כולן.
# HE: near_dup_mask היא רשימת True/False לכל שורה. נשמור אותה: בסוף נחזור אליה לבדיקת רגישות.
key_cols = [c for c in raw.columns if c != "Price (Euro)"]
near_dup_mask = raw.duplicated(subset=key_cols, keep=False)
print("rows in near-duplicate groups:", int(near_dup_mask.sum()))

# HE: groupby מקבצת את השורות לפי התצורה, ו-agg מחשבת לכל קבוצה: כמה שורות, מחיר מינימלי ומקסימלי.
# HE: ratio = מחיר גבוה חלקי מחיר נמוך באותה תצורה. 1.10 אומר שההפרש הוא 10 אחוז.
dup_groups = (raw[near_dup_mask]
              .groupby(key_cols)["Price (Euro)"]
              .agg(rows="count", min_price="min", max_price="max")
              .reset_index())
dup_groups["ratio"] = (dup_groups["max_price"] / dup_groups["min_price"]).round(3)
print("near-duplicate groups:", len(dup_groups))
print("groups where the price differs:", int((dup_groups["ratio"] > 1).sum()))
dup_groups.sort_values("ratio", ascending=False)[
    ["Company", "Product", "RAM (GB)", "Memory", "rows", "min_price", "max_price", "ratio"]
].head(5)

**What section 4 shows**

- **Missing values:** there are no empty cells and no blank text. The only hidden problem is the 208 `Memory` strings with a double space (the two-drive entries, see above). There are no zero or negative numbers.
- **Exact duplicates:** none.
- **Near-duplicates:** 48 rows form 24 pairs that are identical in every column except the price, and in all 24 pairs the prices differ. The largest relative difference is a Lenovo Yoga Book (319 vs 549 euro, a ratio of 1.72). We do not know why the same configuration has different prices (different stores, dates or listing errors).
- **Decision `q_dups`:** we keep these rows and document them. They are not independent observations, so p-values computed on the full table are slightly optimistic. The mask `near_dup_mask` is kept so that, at the end of the analysis, we can check that the conclusions hold without them. This is also listed as a limitation of the report.

## 5. Ranges and units

Decision (`q_audit`, ranges and units): for every numeric column we state the unit and check that the values are plausible for laptops. The plausible ranges below are our own assumptions about typical laptops; they are used to **flag** suspicious values, not to delete them.

In [ ]:
# HE: טבלת יחידות: מה כל עמודה מספרית מודדת ובאיזו יחידה. זה עונה על השאלה "האם היחידות סבירות".
units = pd.DataFrame({
    "meaning": ["screen diagonal", "CPU clock speed", "memory (RAM)", "weight", "price"],
    "unit": ["inch", "GHz", "GB", "kg", "euro"],
}, index=NUMERIC)

# HE: describe() מחשבת סטטיסטיקות תיאוריות. .T הופכת שורות לעמודות, ובוחרים רק מינימום,
# HE: רבעונים (25%, 50% = חציון, 75%) ומקסימום. join מחברת את הטבלה לטבלת היחידות לפי שם העמודה.
summary = raw[NUMERIC].describe().T[["min", "25%", "50%", "75%", "max"]]
units.join(summary)

In [ ]:
# HE: טווחים סבירים למחשבים ניידים: הנחה שלנו (לא עובדה מהנתונים), כדי לסמן ערכים חשודים.
# HE: לכל עמודה זוג (מינימום, מקסימום). הפעולה (raw[c] < lo) | (raw[c] > hi) נותנת True לשורות מחוץ לטווח.
expected = {
    "Inches": (10, 19),
    "CPU_Frequency (GHz)": (0.8, 4.5),
    "RAM (GB)": (2, 64),
    "Weight (kg)": (0.5, 5.0),
    "Price (Euro)": (100, 7000),
}
out_of_range = {c: int(((raw[c] < lo) | (raw[c] > hi)).sum()) for c, (lo, hi) in expected.items()}
print("rows outside the assumed plausible range:", out_of_range)

# HE: מציגים גם את ערכי הקצה בפועל, כדי לראות בעיניים שהם הגיוניים.
print("screen sizes:", sorted(raw["Inches"].unique().tolist()))
print("RAM values and how many laptops each:", raw["RAM (GB)"].value_counts().sort_index().to_dict())

**What section 5 shows**

- Every numeric column is inside the plausible range we assumed: screen 10.1-18.4 inch, CPU 0.9-3.6 GHz, RAM 2-64 GB, weight 0.69-4.7 kg and price 174-6,099 euro (median 989). The units look right: inches, GHz, GB, kg and euro.
- RAM has nine different values. The extreme values are rare: 24 GB appears in 3 laptops and 64 GB in only 1, so any group built on them will be unstable (listed as a limitation).

## 6. Consistency between columns

Decisions: `q_audit` (consistency checks between columns, models and specifications that do not contradict each other) and `q_issues` (implausible values or models are flagged and examined, not deleted automatically).

We have no external source to check that a model really exists on the market, so we can only test **internal consistency**: physical plausibility of the numbers, and whether the specification written inside the model name agrees with the specification columns.

In [ ]:
# HE: בדיקות הגיון פיזיקליות. כל כלל הוא מסכה (True/False לכל שורה) שמסמנת שילוב חשוד.
# HE: & פירושו "וגם". isin בודקת אם הערך נמצא ברשימה.
flags = {
    "small screen (<=13.3 in) but heavy (>2.5 kg)": (raw["Inches"] <= 13.3) & (raw["Weight (kg)"] > 2.5),
    "large screen (>=17 in) but light (<1.5 kg)": (raw["Inches"] >= 17) & (raw["Weight (kg)"] < 1.5),
    "Netbook with 16 GB RAM or more": (raw["TypeName"] == "Netbook") & (raw["RAM (GB)"] >= 16),
    "Gaming/Workstation under 500 euro": raw["TypeName"].isin(["Gaming", "Workstation"]) & (raw["Price (Euro)"] < 500),
    "32 GB RAM or more under 1000 euro": (raw["RAM (GB)"] >= 32) & (raw["Price (Euro)"] < 1000),
}
print("rows flagged by each rule:")
print(pd.Series({name: int(mask.sum()) for name, mask in flags.items()}))

# HE: מתאם בין גודל מסך למשקל: אם הוא חיובי וחזק, המספרים מתיישבים עם הפיזיקה (מסך גדול = כבד יותר).
print("correlation between screen size and weight:", round(raw["Inches"].corr(raw["Weight (kg)"]), 3))

# HE: any_flag הוא True לשורה שסומנה לפחות בכלל אחד. np.logical_or.reduce מחבר את כל המסכות.
any_flag = np.logical_or.reduce(list(flags.values()))
raw.loc[any_flag, ["Company", "Product", "TypeName", "Inches", "RAM (GB)", "Weight (kg)", "Price (Euro)"]]

In [ ]:
# HE: בעמודה Product יש לפעמים מפרט בתוך השם, למשל "X540UA (i3-6006U/4GB/1TB/FHD/Linux)".
# HE: הסדר הרגיל: מעבד / RAM / אחסון. נשווה את ה-RAM והאחסון שבשם לעמודות RAM ו-Memory.
import re

def name_tokens(name):
    # HE: re.findall מחפשת בטקסט את כל ההופעות של התבנית "/מספר GB" או "/מספר TB" ומחזירה רשימה.
    return re.findall(r"/(\d+(?:\.\d+)?)(GB|TB)", name)

def to_gb(value, unit):
    # HE: ממירים ל-GB. הנחה שלנו (כמו ב-st_extra): 1 TB = 1000 GB.
    return float(value) * (1000 if unit == "TB" else 1)

rows = []
for idx, name in raw["Product"].items():
    tokens = name_tokens(name)
    if len(tokens) >= 2:   # HE: רק שמות שיש בהם גם RAM וגם אחסון
        # HE: כל הכוננים שרשומים בעמודה Memory, למשל "128GB SSD + 1TB HDD" -> [128, 1000]
        drives = [to_gb(v, u) for v, u in re.findall(r"([\d.]+)\s*(GB|TB)", raw.at[idx, "Memory"])]
        rows.append({"idx": idx, "name_ram": to_gb(*tokens[0]), "name_storage": to_gb(*tokens[1]),
                     "ram": raw.at[idx, "RAM (GB)"], "drives": drives})

check = pd.DataFrame(rows).set_index("idx")
check["ram_matches"] = check["name_ram"] == check["ram"]
# HE: האחסון שבשם צריך להיות אחד הכוננים (לא בהכרח הראשון), ולכן בודקים "האם נמצא ברשימה".
check["storage_matches"] = [s in d for s, d in zip(check["name_storage"], check["drives"])]
print("model names with a specification inside:", len(check))
print("RAM in the name equals the RAM column:", int(check["ram_matches"].sum()))
print("storage in the name equals one of the drives in Memory:", int(check["storage_matches"].sum()))

In [ ]:
# HE: מציגים את השורות שבהן האחסון שבשם הדגם לא תואם לאף כונן בעמודה Memory, כדי לבדוק אותן בעין.
# HE: ~ הופך True ל-False. join מחברת לטבלה את שם הדגם ואת Memory מהטבלה הגולמית לפי מספר השורה.
mismatch = check[~check["storage_matches"]].join(raw[["Company", "Product", "Memory"]])
mismatch[["Company", "Product", "Memory", "name_storage", "ram", "name_ram"]]

**What section 6 shows**

- **Physical plausibility:** four of the five rules flag nothing. The fifth flags one laptop: a Lenovo ThinkPad Yoga labelled `Netbook` with 16 GB RAM, a 12.5 inch screen, 1.3 kg and a price of 1,650 euro. The label `Netbook` looks questionable for such a machine; we keep the row and flag it. Screen size and weight are strongly correlated (0.83), as expected physically.
- **Model name against the specification columns:** 151 model names carry a specification. The RAM written in the name equals the `RAM (GB)` column in all 151. The storage written in the name equals one of the drives in `Memory` in 148 of them. The 3 remaining rows are Lenovo B51-80 laptops whose name says `1008GB` while `Memory` says `1.0TB Hybrid`: this is the same drive written in two conventions (1,008 GB against 1 TB = 1,000 GB), not a contradiction. It also reminds us that our conversion `1 TB = 1000 GB` (decision `st_extra`) is a convention.
- **Conclusion:** within what we can check, we found no impossible numbers and no model name that contradicts its specification. We cannot verify that a model really exists on the market, because we have no external source; this is a limitation of the audit and goes into the report.

## 7. Issues register and audit summary

Decision `q_issues`: for every problem we found we record the evidence, whether we **fix** it (in task Q2 B) or only **document** it, and where it is handled. All the evidence below is computed from the data, not typed by hand.

First, rare categories: categories with very few laptops cannot be estimated reliably (decision `rare`: group them in a helper column for the analysis, never delete rows).

In [ ]:
# HE: קטגוריה נדירה = פחות מ-10 מחשבים. קטגוריה כזו לא מאפשרת הערכה יציבה (למשל ממוצע מחיר של מחשב אחד).
# HE: value_counts סופרת כמה פעמים מופיע כל ערך. רק הערכים שספירתם קטנה מהסף נשארים ב-rare.
def rare_summary(col, threshold=10):
    counts = raw[col].value_counts()
    rare = counts[counts < threshold]
    return {"categories": len(counts),
            "rare categories (<10 laptops)": len(rare),
            "rows in rare categories": int(rare.sum())}

# HE: בונים טבלה: שורה לכל עמודה קטגוריאלית. .T הופך כך שהעמודות הקטגוריאליות יהיו שורות.
rare_table = pd.DataFrame({c: rare_summary(c) for c in CATEGORICAL}).T
print(rare_table)

# HE: אילו חברות נדירות? (ערכי הספירה מ-value_counts, מסוננים לפי הסף)
companies = raw["Company"].value_counts()
print("rare companies:", companies[companies < 10].to_dict())

In [ ]:
# HE: מחשבים מראש את כל המספרים שבטבלת הבעיות, כדי שכל מספר בטקסט יגיע מהקוד ולא ייכתב ביד.
os_counts = raw["OpSys"].value_counts()
two_drives = raw["Memory"].str.contains("+", regex=False)
has_wxh = raw["ScreenResolution"].str.contains(r"\d+x\d+")

# HE: register היא טבלת הבעיות. כל שורה: בעיה, ראיה מספרית, ההחלטה (תיקון או רק תיעוד) ואיפה מטפלים בה.
# HE: f"..." היא מחרוזת שאפשר לשלב בה ערכים מתוך קוד, בתוך סוגריים מסולסלים.
register = pd.DataFrame([
    {"issue": "Same operating system under two names",
     "evidence": f"macOS ({os_counts['macOS']} rows) and Mac OS X ({os_counts['Mac OS X']} rows)",
     "decision": "fix", "where handled": "Q2 B (os_map)"},
    {"issue": "Memory mixes capacity and storage type",
     "evidence": f"{raw['Memory'].nunique()} different strings; {int(two_drives.sum())} with two drives; GB and TB units",
     "decision": "fix", "where handled": "Q2 B (st_type, Storage_GB_total, has_SSD)"},
    {"issue": "ScreenResolution mixes resolution and panel features",
     "evidence": f"{raw['ScreenResolution'].nunique()} different strings; a WxH resolution in {int(has_wxh.sum())} rows",
     "decision": "fix", "where handled": "Q2 B (Screen_pixels, Touchscreen)"},
    {"issue": "Near-duplicate rows (same configuration, different price)",
     "evidence": f"{int(near_dup_mask.sum())} rows in {len(dup_groups)} groups",
     "decision": "keep and document", "where handled": "sensitivity check at the end of the analysis (q_dups)"},
    {"issue": "Rare categories",
     "evidence": f"{int(rare_table.loc['Company', 'rare categories (<10 laptops)'])} of "
                 f"{int(rare_table.loc['Company', 'categories'])} companies have fewer than 10 laptops",
     "decision": "group in a helper column, never delete rows", "where handled": "Q2 B (rare)"},
    {"issue": "Product holds a model name with a specification inside",
     "evidence": f"{raw['Product'].nunique()} different values; specification inside the name in {len(check)} rows",
     "decision": "not used as an analysis variable", "where handled": "used only for the consistency check in section 6"},
    {"issue": "Implausible values or models",
     "evidence": f"{int(any_flag.sum())} row flagged (a Netbook with 16 GB RAM); no impossible numbers",
     "decision": "flag and keep", "where handled": "examined in the outlier analysis (Q2 D)"},
])
register

### Audit summary

- **Structure:** 1,275 rows and 15 columns (5 numeric, 5 categorical, 5 structured text). A row is a laptop listing: a configuration with one price.
- **Quality:** no missing values and no exact duplicates. 48 rows form 24 near-duplicate pairs (same configuration, different price); we keep them and check at the end that the conclusions hold without them.
- **Types and units:** the numeric columns have numeric types and plausible ranges and units. Text columns hide several pieces of information each: `Memory`, `ScreenResolution`, and in part `Product` and `CPU_Type`.
- **Consistency:** the physical checks flag a single laptop (a `Netbook` with 16 GB RAM). Specifications written in model names agree with the columns (the three exceptions are a unit convention, 1,008 GB against 1 TB).
- **Rare categories:** 11 of 19 companies have fewer than 10 laptops (51 rows), and 3 operating-system labels are rare. Groups this small cannot be estimated reliably.
- **What we fix in Q2 B:** the two names of the same operating system, the parsing of `Memory` and `ScreenResolution`, and the helper column for rare categories. **What we only document:** near-duplicates, the flagged laptop, and the `Product` column.
- **What the audit cannot tell us:** whether the models exist on the market (no external source), when the data was collected, and whether the dataset represents the whole market. These go into the limitations of the report.

# Q2 B: Data cleaning and feature construction

We now build a clean version of the table. The raw table `raw` is never changed: every step works on a copy, and every transformation is validated, because code that runs is not proof that the transformation is right.

| Decision | What we chose |
|---|---|
| `os_map` | Merge only true aliases (`Mac OS X` becomes `macOS`) and add the column `OS_family` for grouping. Windows versions stay separate. |
| `st_type` | `Storage_type` is a canonical combination label, for example `SSD + HDD`. |
| `st_extra` | New variables: `Screen_pixels`, `Storage_GB_total` (1 TB = 1000 GB), `has_SSD`, `CPU_tier`, `Touchscreen`, `log_price`, `Dedicated_GPU`. |
| `rare` | Rare categories are grouped into `Other` in helper columns; no row is deleted. |
| `validate` | Cross-tabulation raw against clean, asserts (no new missing values, nothing unmapped), capacity computed in two ways, and a spot check of 15 random rows. |

## 1. Operating system (`os_map`)

The audit found the same system under two names: `macOS` (13 rows) and `Mac OS X` (8 rows). We merge only true aliases. `Windows 10`, `Windows 7` and `Windows 10 S` are different products, not spelling variants, so they stay separate in `OpSys_clean`; the column `OS_family` groups them for analyses that need larger groups.

The mapping lives in the function `clean_opsys` in `q2_clean.py`. It stops with an error if it meets an operating system it does not know, instead of silently leaving it unmapped.

In [ ]:
# HE: מייבאים את הפונקציה שכתבנו ב-q2_clean.py.
from q2_clean import clean_opsys

# HE: df הוא עותק של הטבלה הגולמית. raw.copy() יוצר עותק שאפשר לשנות בלי לפגוע במקור.
df = raw.copy()

# HE: הפונקציה מחזירה שתי עמודות. השמה כפולה "a, b = ..." שומרת כל אחת בעמודה חדשה.
df["OpSys_clean"], df["OS_family"] = clean_opsys(df["OpSys"])

# HE: טבלת התאמה (crosstab): שורה לכל שם מקורי, עמודה לכל שם חדש, והמספרים הם כמות המחשבים.
# HE: כך רואים בעין שכל שם מקורי הפך לשם הצפוי.
pd.crosstab(df["OpSys"], df["OpSys_clean"])

In [ ]:
# HE: בדיקות אימות (validate). assert עוצר אם משהו לא תקין.
# HE: 1. מספר השורות לא השתנה (לא מחקנו מחשבים).
assert len(df) == len(raw)
# HE: 2. אין ערכים חסרים חדשים בשתי העמודות החדשות.
assert df[["OpSys_clean", "OS_family"]].notna().all().all()
# HE: 3. כל שם מקורי הפך לשם נקי אחד בלבד (בלי "פיצול"). לכל שורה בטבלת ההתאמה צריך להיות ערך אחד גדול מאפס.
table = pd.crosstab(df["OpSys"], df["OpSys_clean"])
assert ((table > 0).sum(axis=1) == 1).all()
# HE: 4. הכינוי נעלם: אין יותר "Mac OS X" בעמודה הנקייה.
assert "Mac OS X" not in set(df["OpSys_clean"])

# HE: ספירת המחשבים בכל מערכת נקייה ובכל משפחה.
print(df["OpSys_clean"].value_counts())
print(df["OS_family"].value_counts())

**What the cross-tabulation shows**

- `Mac OS X` (8 rows) and `macOS` (13 rows) are now one category, `macOS`, with 21 laptops. Every other operating system maps to itself, so the merge changed exactly these 8 rows.
- The Windows versions stay separate in `OpSys_clean` (`Windows 10`: 1,048, `Windows 7`: 45, `Windows 10 S`: 8) and are grouped in `OS_family` (`Windows`: 1,101).
- `No OS` (66 laptops) is a real category (a laptop sold without a system), not a missing value, so it is kept.
- `Android` (2) and `Windows 10 S` (8) are rare categories; they are handled by the `rare` decision later.

A small test of the function itself (decision `g_verify`: a function is tested on an input whose answer we know before we trust it on the real data).

In [ ]:
# HE: בדיקה קטנה של הפונקציה על קלט שאנחנו יודעים מראש מה התשובה שלו.
test = pd.Series(["Mac OS X", "macOS", "Windows 7"])
clean_test, family_test = clean_opsys(test)
assert list(clean_test) == ["macOS", "macOS", "Windows 7"]
assert list(family_test) == ["macOS", "macOS", "Windows"]

# HE: מערכת הפעלה לא מוכרת חייבת לגרום לשגיאה (assert), ולא לעבור בשקט.
# HE: try/except תופס את השגיאה כדי שנוכל לבדוק שהיא אכן קרתה.
raised = False
try:
    clean_opsys(pd.Series(["BeOS"]))
except AssertionError:
    raised = True
assert raised, "an unknown operating system must be rejected"
print("clean_opsys passed its tests")

## 2. Storage: `Storage_type`, `Storage_GB_total` and `has_SSD`

The `Memory` column holds one or two drives in a single string, for example `128GB SSD +  1TB HDD`. Decisions: `st_type` (the type is a canonical combination label) and `st_extra` (total capacity with 1 TB = 1000 GB, and a flag for SSD).

How we parse each string (function `parse_memory` in `q2_clean.py`):

- A drive is a number, a unit (`GB` or `TB`) and a type (`SSD`, `HDD`, `Flash Storage` or `Hybrid`), found with a regular expression that also accepts decimals such as `1.0TB` and the double spaces we found in the audit.
- `Storage_type` lists the drive types present in a fixed order, so `SSD + HDD` is always written the same way. Two drives of the same type give one type (`256GB SSD + 256GB SSD` becomes `SSD`).
- `Storage_GB_total` adds the capacities of all drives.
- `has_SSD` is true only when an `SSD` is present. `Flash Storage` is not counted as SSD, because it is a different (smaller, embedded) kind of memory; this is our reading of the data.
- The function stops with an error if any text remains that it did not understand, so a new format cannot slip through silently.

In [ ]:
from q2_clean import parse_memory, parse_memory_series

# HE: מפעילים את הפירוק על כל עמודת Memory, ומצמידים את שלוש העמודות החדשות לעותק df (לפי מספר השורה).
# HE: pd.concat([...], axis=1) מדביקה טבלאות זו לצד זו.
storage = parse_memory_series(df["Memory"])
df = pd.concat([df, storage], axis=1)

# HE: טבלת התאמה מלאה: כל מחרוזת Memory ייחודית (יש 39) והתוצאה שלה. 39 שורות אפשר לעבור עליהן בעין.
# HE: drop_duplicates משאירה כל צירוף פעם אחת, ו-sort_values ממיינת לפי המחרוזת המקורית.
mapping = (df[["Memory", "Storage_type", "Storage_GB_total", "has_SSD"]]
           .drop_duplicates().sort_values("Memory").reset_index(drop=True))
print(len(mapping), "different Memory strings")
mapping

In [ ]:
# HE: כמה מחשבים בכל תווית אחסון, וכמה עם SSD.
print(df["Storage_type"].value_counts())
print("laptops with an SSD:", int(df["has_SSD"].sum()))
print(df["Storage_GB_total"].describe())

In [ ]:
# HE: בדיקת אימות (validate): מחשבים את הקיבולת בדרך שנייה, בלתי תלויה, ומשווים.
# HE: הדרך השנייה לא משתמשת בביטוי הרגולרי: מפצלים את המחרוזת לפי "+", ולכל חלק לוקחים את המילה הראשונה
# HE: (למשל "128GB") וחותכים את היחידה מהסוף. אם שתי הדרכים נותנות אותו מספר, הסיכוי לטעות קטן.
def capacity_second_way(memory):
    total = 0.0
    for part in memory.split("+"):
        word = part.split()[0]                 # HE: המילה הראשונה בחלק, למשל "1.0TB"
        if word.endswith("TB"):
            total += float(word[:-2]) * 1000
        elif word.endswith("GB"):
            total += float(word[:-2])
        else:
            raise ValueError(f"unexpected unit in {memory!r}")
    return total

second = df["Memory"].apply(capacity_second_way)
# HE: np.isclose משווה מספרים עשרוניים בצורה בטוחה (בלי שגיאות עיגול קטנות).
assert np.isclose(df["Storage_GB_total"], second).all(), "the two capacity computations disagree"

# HE: עוד בדיקות: אין ערכים חסרים, הקיבולת חיובית, מספר השורות לא השתנה, וכל תווית אחסון היא אחת מהמותרות.
assert df[["Storage_type", "Storage_GB_total", "has_SSD"]].notna().all().all()
assert (df["Storage_GB_total"] > 0).all()
assert len(df) == len(raw)
print("capacity computed in two independent ways: identical for all", len(df), "rows")

A test of the parser on strings whose answer we know in advance (decision `g_verify`):

In [ ]:
# HE: בדיקות קטנות לפונקציה על קלט שהתשובה שלו ידועה מראש. כל שורה אומרת: על הקלט הזה צריך לצאת בדיוק כך.
cases = {
    "128GB SSD +  1TB HDD": ("SSD + HDD", 1128.0, True),
    "1.0TB Hybrid": ("Hybrid", 1000.0, False),
    "256GB SSD +  256GB SSD": ("SSD", 512.0, True),          # HE: שני SSD נחשבים סוג אחד
    "32GB Flash Storage": ("Flash Storage", 32.0, False),
    # HE: הסדר הקבוע שהגדרנו בתווית הוא SSD, אחר כך HDD, אחר כך Flash Storage, אחר כך Hybrid. לכן Flash אחרי HDD.
    "64GB Flash Storage +  1TB HDD": ("HDD + Flash Storage", 1064.0, False),
    "1TB HDD +  1TB HDD": ("HDD", 2000.0, False),
}
for text, expected in cases.items():
    assert parse_memory(text) == expected, (text, parse_memory(text), expected)

# HE: טקסט שלא מוכר חייב להיכשל עם שגיאה.
raised = False
try:
    parse_memory("5GB Tape")
except AssertionError:
    raised = True
assert raised, "an unknown drive type must be rejected"
print("parse_memory passed its tests")

**What section 2 shows**

- All 39 different `Memory` strings are parsed, and the full mapping table lets us check every one by eye. Examples: `128GB SSD +  1TB HDD` becomes `SSD + HDD` with 1,128 GB, `1.0TB Hybrid` becomes `Hybrid` with 1,000 GB, and `512GB SSD +  512GB SSD` becomes `SSD` with 1,024 GB.
- Storage types: `SSD` 635, `HDD` 359, `SSD + HDD` 200, `Flash Storage` 70, `Hybrid` 8, `SSD + Hybrid` 2 and `HDD + Flash Storage` 1. These agree with the audit: 10 laptops with Hybrid (8 + 2), 71 with Flash Storage (70 + 1), and 208 with two drives (200 + 2 + 1 of different types and 5 with two drives of the same type).
- 837 laptops have an SSD (635 + 200 + 2). Total capacity ranges from 8 GB to 2,512 GB, with a median of 500 GB.
- The combinations with very few laptops (`SSD + Hybrid` with 2, `HDD + Flash Storage` with 1, `Hybrid` with 8) cannot be compared reliably; the `rare` decision groups them in a helper column for the analysis, and the rows stay.
- The capacity computed by the parser and the capacity computed by a second, independent method (splitting on `+`) are identical for all 1,275 rows.

**A test caught a mistake of ours.** In the first version of the test cell we expected `Flash Storage + HDD` for the string `64GB Flash Storage +  1TB HDD`, but the parser returned `HDD + Flash Storage`. The parser was following the fixed order we defined (`SSD`, `HDD`, `Flash Storage`, `Hybrid`), so the expectation in the test was wrong, not the function. We corrected the test and kept the fixed order, which is what makes the labels consistent.

## 3. Screen, processor, price and graphics variables

Decision `st_extra` asks for five more variables. Each one is a function in `q2_clean.py` that takes a column and returns a new column:

| New variable | How it is computed | Why we need it |
|---|---|---|
| `Screen_pixels` | width times height, taken from the `WxH` resolution at the end of `ScreenResolution` | resolution raises the price and is linked to the laptop type |
| `Touchscreen` | `ScreenResolution` mentions `Touchscreen` | a touchscreen raises the price |
| `CPU_tier` | `Core i3`, `Core i5`, `Core i7`, or `Other` (Celeron, Pentium, Atom, AMD, Core M...), from the start of `CPU_Type` | the processor is the main confounder of the RAM-price relation: laptops with a lot of RAM also tend to have a strong processor |
| `log_price` | natural logarithm of the price | prices are right-skewed, and effects on price are multiplicative (decision `tr`) |
| `Dedicated_GPU` | the graphics company is Nvidia or AMD (True), Intel or ARM (False) | a dedicated card raises the price |

`Dedicated_GPU` is an approximation: it is decided by the company only, and some AMD cards are integrated graphics. We keep the decision and state this as a limitation of the analysis.

In [ ]:
from q2_clean import screen_pixels, has_touchscreen, cpu_tier, log_price, dedicated_gpu

# HE: מפעילים כל פונקציה על העמודה המתאימה ושומרים בעמודה חדשה בעותק df (הטבלה הגולמית raw לא משתנה).
df["Screen_pixels"] = screen_pixels(df["ScreenResolution"])
df["Touchscreen"] = has_touchscreen(df["ScreenResolution"])
df["CPU_tier"] = cpu_tier(df["CPU_Type"])
df["log_price"] = log_price(df["Price (Euro)"])
df["Dedicated_GPU"] = dedicated_gpu(df["GPU_Company"])

# HE: מציגים כמה שורות של העמודה המקורית לצד העמודה החדשה, כדי לראות בעין שהתוצאה הגיונית.
df[["ScreenResolution", "Screen_pixels", "Touchscreen", "CPU_Type", "CPU_tier",
    "GPU_Company", "Dedicated_GPU", "Price (Euro)", "log_price"]].head(6)

In [ ]:
# HE: בודקים את דרגת המעבד: כמה מחשבים בכל דרגה, ומה נכנס לקבוצה "Other".
print(df["CPU_tier"].value_counts())
# HE: groupby מקבצת לפי דרגה; nunique סופרת כמה דגמי מעבד שונים נכנסו לכל דרגה.
print(df.groupby("CPU_tier")["CPU_Type"].nunique())
# HE: אילו דגמים נכנסו ל-Other (הנפוצים ביותר), ואיזו חברת מעבדים: זה השק המעורב.
print(df.loc[df["CPU_tier"] == "Other", "CPU_Type"].value_counts().head(8))
print(df.loc[df["CPU_tier"] == "Other", "CPU_Company"].value_counts())

In [ ]:
# HE: סטטיסטיקות של שאר העמודות החדשות.
print("pixels: min", df["Screen_pixels"].min(), " max", df["Screen_pixels"].max())
print("touchscreen laptops:", int(df["Touchscreen"].sum()))
print("dedicated GPU laptops:", int(df["Dedicated_GPU"].sum()), "of", len(df))
print(df.groupby("GPU_Company")["Dedicated_GPU"].agg(["count", "sum"]))
# HE: skew מודד עד כמה ההתפלגות מוטה לצד אחד. 0 הוא סימטרי. אחרי הלוג צפוי מספר קרוב לאפס.
print("skewness of the price:", round(df["Price (Euro)"].skew(), 2), " of log_price:", round(df["log_price"].skew(), 2))

In [ ]:
# HE: בדיקות אימות (validate). 1. אין ערכים חסרים בעמודות החדשות. 2. מספר השורות לא השתנה.
new_cols = ["Screen_pixels", "Touchscreen", "CPU_tier", "log_price", "Dedicated_GPU"]
assert df[new_cols].notna().all().all()
assert len(df) == len(raw)
# HE: 3. פיקסלים חיוביים. 4. הלוג הפיך: exp של הלוג מחזיר את המחיר המקורי (בדרך שנייה בודקים שלא פגענו בנתון).
assert (df["Screen_pixels"] > 0).all()
assert np.isclose(np.exp(df["log_price"]), df["Price (Euro)"]).all()
# HE: 5. דגם המעבד "Core i5 7200U" באמת הפך ל-"Core i5".
assert (df.loc[df["CPU_Type"] == "Core i5 7200U", "CPU_tier"] == "Core i5").all()
print("all checks passed")

Tests of the functions on small inputs whose answer we know in advance (decision `g_verify`):

In [ ]:
# HE: בדיקות קטנות לפונקציות על קלט שהתשובה שלו ידועה מראש.
res_test = pd.Series(["IPS Panel Full HD 1920x1080", "1366x768", "IPS Panel Touchscreen 1920x1200"])
assert list(screen_pixels(res_test)) == [2073600, 1049088, 2304000]
assert list(has_touchscreen(res_test)) == [False, False, True]

assert list(cpu_tier(pd.Series(["Core i5 7200U", "Core i7", "Celeron Dual Core N3060", "Core M 6Y75"]))) == \
       ["Core i5", "Core i7", "Other", "Other"]
assert list(dedicated_gpu(pd.Series(["Nvidia", "Intel", "AMD", "ARM"]))) == [True, False, True, False]
assert np.isclose(log_price(pd.Series([np.e]))[0], 1.0)

# HE: קלט לא תקין חייב להיכשל עם שגיאה. פונקציה שמקבלת חברת כרטיס לא מוכרת צריכה להיעצר.
raised = False
try:
    dedicated_gpu(pd.Series(["Matrox"]))
except AssertionError:
    raised = True
assert raised, "an unknown GPU company must be rejected"
print("the five functions passed their tests")

**What section 3 shows**

- **Screen:** the resolution was found in all 1,275 rows. Pixels range from 1,049,088 (1366x768) to 8,294,400 (4K), and 188 laptops have a touchscreen, as in the audit.
- **Processor tier:** `Core i7` 515, `Core i5` 423, `Core i3` 134 and `Other` 203. The `Other` group is a mixed bag: 57 different processor models (Celeron, Pentium, Atom, AMD A-series, Core M...), of which 142 laptops have an Intel processor, 60 an AMD processor and 1 a Samsung one. It is enough to separate "strong" from "weak" processors, but not a fine description.
- **Price:** the skewness drops from 1.51 (raw price) to -0.18 (`log_price`), and taking `exp` of the log returns the original price for every row.
- **Dedicated graphics:** 570 of 1,275 laptops (Nvidia 396 and AMD 174). This is an approximation by company only: the sixth row of the table above is an AMD A9 laptop priced at 400 euro whose AMD graphics are almost certainly integrated, yet it is marked as dedicated. We keep the decision, as agreed, and state this as a limitation: the effect of a dedicated card is somewhat overstated for AMD laptops.

## 4. Rare categories and flags

Decisions: `rare` (categories with fewer than 10 laptops are grouped into `Other` in **helper columns**, and no row is deleted), `st_type` (rare storage combinations are grouped in the analysis only), `q_dups` (near-duplicates are kept and flagged) and `q_issues` (implausible rows are flagged, not removed).

A category with a handful of laptops cannot be estimated reliably, so analyses that compare groups use the grouped helper columns, while the original columns stay untouched.

In [ ]:
from q2_clean import group_rare, flag_near_duplicates, flag_suspicious

# HE: עמודות עזר: לכל עמודה קטגוריאלית מוסיפים עמודה חדשה שבה הקטגוריות הנדירות הפכו ל-"Other".
# HE: העמודות המקוריות נשארות כמו שהן. group_rare מחזירה עמודה חדשה ולא מוחקת שורות.
df["Company_grouped"] = group_rare(df["Company"])
df["OpSys_grouped"] = group_rare(df["OpSys_clean"])
df["CPU_Company_grouped"] = group_rare(df["CPU_Company"])
df["GPU_Company_grouped"] = group_rare(df["GPU_Company"])
df["Storage_type_grouped"] = group_rare(df["Storage_type"], label="Other storage")

# HE: טבלת השוואה: לכל עמודה, כמה קטגוריות היו, כמה נשארו אחרי הקיבוץ, וכמה שורות נכנסו ל-Other.
pairs = {"Company": "Company_grouped", "OpSys_clean": "OpSys_grouped", "CPU_Company": "CPU_Company_grouped",
         "GPU_Company": "GPU_Company_grouped", "Storage_type": "Storage_type_grouped"}
summary_rare = pd.DataFrame({
    "categories before": {c: df[c].nunique() for c in pairs},
    "categories after": {c: df[g].nunique() for c, g in pairs.items()},
    "rows in Other": {c: int(df[g].isin(["Other", "Other storage"]).sum()) for c, g in pairs.items()},
})
summary_rare

In [ ]:
# HE: בדיקות אימות (validate) לקיבוץ הנדירים.
# HE: 1. לא מחקנו שורות. 2. אין ערכים חסרים בעמודות העזר.
helper_cols = list(pairs.values())
assert len(df) == len(raw)
assert df[helper_cols].notna().all().all()
# HE: 3. בכל עמודת עזר, כל קטגוריה שנשארה (חוץ מ-Other) מופיעה לפחות 10 פעמים.
for g in helper_cols:
    counts = df[g].value_counts()
    kept = counts[~counts.index.isin(["Other", "Other storage"])]
    assert (kept >= 10).all(), g
# HE: 4. בדרך שנייה: מספר השורות ב-Other שווה לסכום הקטגוריות הנדירות שמצאנו ישירות, בלי הפונקציה.
for c, g in pairs.items():
    counts = df[c].value_counts()
    assert int(df[g].isin(["Other", "Other storage"]).sum()) == int(counts[counts < 10].sum()), c
print("rare-category grouping passed its checks")

# HE: אילו חברות נכנסו ל-Other, ואילו נשארו.
print("kept companies:", sorted(df["Company_grouped"].unique()))

In [ ]:
# HE: סימונים. עמודה בוליאנית (True/False) לכל סימון. לא מוחקים שום שורה.
# HE: key_cols: כל העמודות המקוריות חוץ מהמחיר. חשוב: לא כוללים את העמודות החדשות (כי log_price נגזר מהמחיר).
key_cols = [c for c in raw.columns if c != "Price (Euro)"]
df["near_duplicate"] = flag_near_duplicates(df, key_cols)
df["suspicious"], rule_masks = flag_suspicious(df)

print("near-duplicate rows:", int(df["near_duplicate"].sum()))
print("suspicious rows:", int(df["suspicious"].sum()))
# HE: אימות בדרך שנייה: הסימונים צריכים להיות זהים למה שחישבנו עצמאית בביקורת (near_dup_mask ו-any_flag).
assert (df["near_duplicate"] == near_dup_mask).all()
assert (df["suspicious"].to_numpy() == any_flag).all()
print("the flags are identical to the independent computations of the audit")
df.loc[df["suspicious"], ["Company", "Product", "TypeName", "RAM (GB)", "Price (Euro)"]]

Tests of the two functions on small inputs whose answer we know in advance (decision `g_verify`):

In [ ]:
# HE: בדיקות קטנות. "a" מופיעה 10 פעמים, "b" פעם אחת: כשהסף הוא 3, "b" נדירה והופכת ל-Other.
small = pd.Series(["a"] * 10 + ["b"])
assert list(group_rare(small, min_count=3).unique()) == ["a", "Other"]
assert list(small) == ["a"] * 10 + ["b"]          # HE: המקור לא השתנה

# HE: שתי שורות זהות בעמודה x ושונות במחיר = כמעט-כפולות. שורה שלישית שונה.
tiny = pd.DataFrame({"x": [1, 1, 2], "Price (Euro)": [10.0, 12.0, 10.0]})
assert list(flag_near_duplicates(tiny, ["x"])) == [True, True, False]

# HE: קיבוץ לא יכול לשכתב קטגוריה אמיתית בשם Other. הפונקציה חייבת לעצור.
raised = False
try:
    group_rare(pd.Series(["Other", "Other", "x"]), min_count=3)
except AssertionError:
    raised = True
assert raised, "a real category called Other must be rejected"
print("group_rare and flag_near_duplicates passed their tests")

**What section 4 shows**

- **Companies:** 19 companies become 9 groups: 8 companies with at least 21 laptops (Acer, Apple, Asus, Dell, HP, Lenovo, MSI, Toshiba) and `Other` with the 11 companies that have fewer than 10 laptops (51 rows in total). No laptop is removed.
- **Operating system:** after merging `Mac OS X` into `macOS` (21 laptops), only two labels remain rare (`Windows 10 S` with 8 and `Android` with 2, 10 rows). The audit, which looked at the raw column, counted three rare labels (18 rows). The order matters: cleaning first changes which categories are rare.
- **Processor and graphics company:** one laptop each (a Samsung processor and an ARM graphics unit) goes to `Other`.
- **Storage type:** `Hybrid` (8), `SSD + Hybrid` (2) and `HDD + Flash Storage` (1) are grouped as `Other storage` (11 rows), leaving `SSD`, `HDD`, `SSD + HDD` and `Flash Storage`.
- **Flags:** 48 near-duplicate rows and 1 suspicious row, exactly the same rows as in the independent computations of the audit (`near_dup_mask` and `any_flag`), checked with `assert`. The flagged laptop is the Lenovo ThinkPad Yoga labelled `Netbook` with 16 GB RAM.